# Module-06: Guided Lab

In [ ]:
# Install the NLTK library (Natural Language Toolkit).
# NLTK is used for working with human language data, such as tokenizing and processing text.
!pip install nltk

# Install the spaCy library.
# spaCy is another powerful NLP library used for tokenization, part-of-speech tagging,
!pip install spacy

# Download the small English language model for spaCy.
# This model (en_core_web_sm) contains vocabulary, grammar, and statistical data
!python -m spacy download en_core_web_sm

This code shows Implementing POS Tagging in Python using NLTK. It downloads the necessary NLTK resources, tokenizes a sample sentence into individual words, and then uses pos_tag() to assign a part-of-speech label (like noun, verb, adjective) to each word. Finally, it prints the list of (word, tag) pairs so you can see how each word in the sentence is grammatically classified.

In [ ]:
# Import the NLTK library for Natural Language Processing tasks
import nltk

# Import two helpful functions from NLTK:
# - word_tokenize: splits a sentence into individual words and punctuation
# - pos_tag: assigns a Part Of Speech (POS) label to each word (noun, verb, etc.)
from nltk import word_tokenize, pos_tag

# Download the tokenizer data that NLTK needs to split text into words.
# This only needs to be done the first time on a new computer.
nltk.download('punkt')

# In some environments, this extra resource is also needed for tokenization.
nltk.download('punkt_tab')

# Download the POS tagger model.
# "averaged_perceptron_tagger_eng" is a pre trained model that knows how to label words
# with parts of speech like NN (noun), VB (verb), JJ (adjective), etc.
nltk.download('averaged_perceptron_tagger_eng')

# ---------------------------------------
# 1. SAMPLE TEXT
# ---------------------------------------

# This is the text we will analyze.
text = "Natural Language Processing with Python is fascinating."

# ---------------------------------------
# 2. TOKENIZE THE TEXT INTO WORDS
# ---------------------------------------

# Use word_tokenize() to split the sentence into a list of tokens.
# A token is usually a word or punctuation.
tokens = word_tokenize(text)

# ---------------------------------------
# 3. PART OF SPEECH (POS) TAGGING
# ---------------------------------------

# Use pos_tag() to assign a part of speech label to each token.
# The result is a list of (word, tag) pairs.
# Example: [("Natural", "JJ"), ("Language", "NN"), ...]
pos_tags = pos_tag(tokens)

# ---------------------------------------
# 4. PRINT THE RESULT
# ---------------------------------------

print("POS Tags:")
# This will show each word together with its POS tag.
print(pos_tags)


the Treebank corpus as the gold standard, uses NLTK’s PerceptronTagger to predict tags for the same sentences, and then flattens both the true and predicted tags into simple lists. Using these lists, it calculates accuracy, precision, recall, and F1 score so you can see how well the POS tagger performs on real annotated data.

In [ ]:
# Import pos_tag (not directly used here, but related to POS tagging in NLTK)
from nltk import pos_tag

# Import the treebank corpus from NLTK.
# The treebank corpus contains sentences that are already POS-tagged,
# and we will use it as "gold standard" (correct answers) for evaluation.
from nltk.corpus import treebank

# Import evaluation metrics from scikit-learn:
# - accuracy_score: how many tags we got correct overall
# - precision_score, recall_score, f1_score: more detailed evaluation of quality
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Import NLTK itself to access taggers and corpora
import nltk

# Download the treebank corpus (only needs to be done once per environment)
nltk.download('treebank')

# ---------------------------------------
# 1. LOAD AND PREPARE TEST DATA
# ---------------------------------------

# treebank.tagged_sents() returns a list of sentences,
# where each sentence is a list of (word, tag) pairs.
# We take sentences from index 3000 to the end as our test set.
test_data = treebank.tagged_sents()[3000:]

# Extract just the words from each sentence for input to the tagger.
# Example: [[("The","DT"),("dog","NN")], ...] -> [["The","dog"], ...]
test_sentences = [[word for word, tag in sent] for sent in test_data]

# Extract just the correct tags from each sentence to use as our "gold standard".
# Example: [[("The","DT"),("dog","NN")], ...] -> [["DT","NN"], ...]
gold_standard = [[tag for word, tag in sent] for sent in test_data]

# ---------------------------------------
# 2. TAG SENTENCES WITH A PRE-TRAINED TAGGER
# ---------------------------------------

# Create a PerceptronTagger, which is an NLTK POS tagger based on the perceptron algorithm.
tagger = nltk.PerceptronTagger()

# Use the tagger to assign tags to each sentence in the test set.
# tagger.tag(sent) returns a list of (word, tag) pairs.
predicted_tags = [tagger.tag(sent) for sent in test_sentences]

# Keep only the predicted tags, drop the words.
# Example: [[("The","DT"),("dog","NN")], ...] -> [["DT","NN"], ...]
predicted_tags = [[tag for word, tag in sent] for sent in predicted_tags]

# ---------------------------------------
# 3. FLATTEN LISTS TO COMPUTE METRICS
# ---------------------------------------

# The metrics functions expect 1D lists of labels, not nested lists.
# So we "flatten" the list of sentences into one long list of tags.

# Flatten the gold standard tags:
# [["DT","NN"],["PRP","VBP"], ...] -> ["DT","NN","PRP","VBP", ...]
gold_standard_flat = [tag for sent in gold_standard for tag in sent]

# Flatten the predicted tags in the same way.
predicted_tags_flat = [tag for sent in predicted_tags for tag in sent]

# ---------------------------------------
# 4. COMPUTE EVALUATION METRICS
# ---------------------------------------

# Accuracy: proportion of tags that we predicted correctly.
accuracy = accuracy_score(gold_standard_flat, predicted_tags_flat)

# Precision: of all tags we predicted as a certain label, how many were correct?
# We use "weighted" average to handle different tag frequencies.
precision = precision_score(gold_standard_flat, predicted_tags_flat, average='weighted')

# Recall: of all true tags of a certain label, how many did we find correctly?
recall = recall_score(gold_standard_flat, predicted_tags_flat, average='weighted')

# F1 score: harmonic mean of precision and recall (a combined score).
f1 = f1_score(gold_standard_flat, predicted_tags_flat, average='weighted')

# ---------------------------------------
# 5. PRINT THE RESULTS
# ---------------------------------------

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


This code demonstrates Training Custom POS Taggers in Python using NLTK and the Treebank corpus. It first splits tagged sentences into a training set and a test set, then trains a UnigramTagger that tags each word based only on how it was seen in the training data. Next, it trains a BigramTagger that uses both the current word and the previous word, with a backoff to the unigram tagger when needed. Finally, it evaluates both taggers on the test data and prints their accuracies so you can compare how much context helps improve POS tagging performance.

In [ ]:
# Import NLTK (Natural Language Toolkit) for NLP tasks
import nltk

# Import two types of taggers:
# - UnigramTagger: tags each word based on how it was most commonly tagged in training
# - BigramTagger: tags words based on the current word AND the previous word
from nltk.tag import UnigramTagger, BigramTagger

# Import the treebank corpus, which contains sentences already labeled with POS tags
from nltk.corpus import treebank

# Download the treebank corpus (only needs to be done once on a new computer)
nltk.download('treebank')

# ---------------------------------------
# 1. LOAD TRAINING AND TEST DATA
# ---------------------------------------

# treebank.tagged_sents() returns a list of sentences.
# Each sentence is a list of (word, tag) pairs, e.g. [("The","DT"), ("dog","NN"), ...]
#
# Here we split the data into:
# - train_data: first 3000 sentences (used to train the taggers)
# - test_data: remaining sentences (used to evaluate how well they perform)
train_data = treebank.tagged_sents()[:3000]
test_data = treebank.tagged_sents()[3000:]

# ---------------------------------------
# 2. TRAIN A UNIGRAM TAGGER
# ---------------------------------------

# A UnigramTagger looks at each word by itself (no context).
# It learns the most likely tag for each word from the training data.
unigram_tagger = UnigramTagger(train_data)

# Evaluate how well the UnigramTagger does on the test data.
# .evaluate() returns accuracy: the fraction of words tagged correctly.
accuracy = unigram_tagger.evaluate(test_data)
print("Unigram Tagger Accuracy:", accuracy)

# ---------------------------------------
# 3. TRAIN A BIGRAM TAGGER WITH BACKOFF
# ---------------------------------------

# A BigramTagger considers the current word AND the previous word to decide the tag.
# It uses patterns like ("previous_tag", "current_word") to predict the current tag.
#
# backoff=unigram_tagger means:
#   If the BigramTagger doesn't know how to tag a word (not seen in training),
#   it will "back off" and ask the UnigramTagger for a guess instead.
bigram_tagger = BigramTagger(train_data, backoff=unigram_tagger)

# Evaluate how well the BigramTagger (with backoff) does on the same test data.
accuracy = bigram_tagger.evaluate(test_data)
print("Bigram Tagger Accuracy:", accuracy)


This code shows Implementing NER in Python using spaCy. It loads a pre-trained English model, processes a sample sentence, and then loops through doc.ents to find and print all named entities (like companies, locations, and money amounts) along with their labels (such as ORG, GPE, MONEY). This lets you automatically detect important real-world items in text.

In [ ]:
# Import the spaCy library.
# spaCy is a powerful NLP (Natural Language Processing) library
# that can find things like names, places, and organizations in text.
import spacy

# Load the pre-trained small English model.
# This model knows basic English grammar, vocabulary, and entities.
# Make sure you have installed it first with:
#   python -m spacy download en_core_web_sm
nlp = spacy.load('en_core_web_sm')

# ---------------------------------------
# 1. SAMPLE TEXT
# ---------------------------------------

# This is the sentence we want to analyze.
text = "Apple is looking at buying U.K. startup for $1 billion."

# ---------------------------------------
# 2. PROCESS THE TEXT
# ---------------------------------------

# Pass the text to the spaCy model.
# The result 'doc' contains tokens, part-of-speech tags, entities, etc.
doc = nlp(text)

# ---------------------------------------
# 3. EXTRACT AND PRINT NAMED ENTITIES
# ---------------------------------------

# Named entities are real-world things like:
# - companies (Apple)
# - locations (U.K.)
# - money amounts ($1 billion)
print("Named Entities:")

# doc.ents is a list of all named entities found in the text.
for ent in doc.ents:
    # ent.text  = the actual text of the entity (e.g., "Apple")
    # ent.label_ = the type of entity (e.g., ORG for organization, GPE for location, MONEY for money)
    print(ent.text, ent.label_)


This code demonstrates Evaluating NER Systems by comparing a model’s predicted named entities with the correct, human-annotated entities. It defines a set of true entities and a set of predicted entities, then uses scikit-learn’s evaluation metrics—precision, recall, and F1 score—to measure how accurate the model is. Precision checks how many predicted entities were correct, recall checks how many real entities the model successfully found, and the F1 score combines both into one overall quality measure. The results help determine how well an NER system is performing.

In [ ]:
# Import evaluation metrics from scikit-learn.
# These functions help us measure how good our model's predictions are.
from sklearn.metrics import precision_score, recall_score, f1_score

# ---------------------------------------
# 1. TRUE (CORRECT) ENTITIES
# ---------------------------------------

# This list represents the "gold standard" entities.
# These are the entities that a human has manually labeled as correct in the text.
true_entities = ["Apple", "U.K.", "startup", "$1 billion"]

# ---------------------------------------
# 2. PREDICTED ENTITIES (MODEL OUTPUT)
# ---------------------------------------

# This list represents the entities found by our NER (Named Entity Recognition) system.
# In this example, some of them are slightly different from the true entities.
predicted_entities = ["Apple", "UK", "startup", "$1B"]

# ---------------------------------------
# 3. CALCULATE PRECISION, RECALL, AND F1
# ---------------------------------------

# precision_score:
#   Of all the entities the model predicted, how many were correct?
# recall_score:
#   Of all the true entities, how many did the model find?
# f1_score:
#   The harmonic mean of precision and recall (a single combined score).
#
# average='micro' means:
#   Treat every prediction equally and compute precision/recall
#   over all items together (not per class).
precision = precision_score(true_entities, predicted_entities, average='micro')
recall = recall_score(true_entities, predicted_entities, average='micro')
f1 = f1_score(true_entities, predicted_entities, average='micro')

# ---------------------------------------
# 4. PRINT THE RESULTS
# ---------------------------------------

print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")


This code demonstrates training a custom NER model in spaCy and Evaluating NER Systems in a simple way. It creates a blank English pipeline, adds a new NER component, and teaches it a custom label "GADGET" using a small set of training sentences where devices like iPhone and iPad Pro are marked as entities. The training loop updates the model over several epochs and prints the training loss so you can see if it’s learning. Finally, the code evaluates the NER system informally by running the trained model on a new sentence and printing out the detected entities and their labels, letting you check whether it correctly recognizes the gadget in unseen text.


In [ ]:
# Import spaCy, a powerful NLP (Natural Language Processing) library
import spacy

# Import helper classes and functions from spaCy:
# - DocBin: (not used in this example, but often used to store training docs efficiently)
# - Example: used to create training examples for spaCy
# - minibatch, compounding: used to create mini-batches of data during training
from spacy.tokens import DocBin
from spacy.training import Example
from spacy.util import minibatch, compounding

# ---------------------------------------
# 1. CREATE A BLANK ENGLISH MODEL
# ---------------------------------------

# Create a blank spaCy pipeline for English.
# This model starts with no components (no tokenizer, tagger, NER, etc. pre-trained).
nlp = spacy.blank("en")

# ---------------------------------------
# 2. ADD A NEW NER COMPONENT
# ---------------------------------------

# Add a Named Entity Recognizer (NER) component to the pipeline.
# "ner" is the name of the built-in entity recognizer in spaCy.
ner = nlp.add_pipe("ner")

# Tell the NER component about a new label we want it to learn.
# In this case, "GADGET" will be used for things like "iPhone", "iPad", etc.
ner.add_label("GADGET")

# ---------------------------------------
# 3. DEFINE TRAINING DATA
# ---------------------------------------

# TRAIN_DATA is a list of (text, annotations) pairs.
# Each "annotations" dictionary has an "entities" key.
# "entities" is a list of tuples: (start_char, end_char, label)
# where start_char and end_char are character positions in the text.
TRAIN_DATA = [
    ("Apple is releasing a new iPhone.", {"entities": [(26, 32, "GADGET")]}),
    ("The new iPad Pro is amazing.", {"entities": [(8, 16, "GADGET")]}),
]

# ---------------------------------------
# 4. CONVERT TRAINING DATA TO spaCy EXAMPLES
# ---------------------------------------

examples = []

# Loop over each (text, annotations) pair in the training data
for text, annotations in TRAIN_DATA:
    # Create a spaCy Doc object from the raw text
    doc = nlp.make_doc(text)

    # Create an Example object that combines the Doc and the annotations
    # Example.from_dict() tells spaCy which spans in the doc are entities
    example = Example.from_dict(doc, annotations)

    # Add the example to our list
    examples.append(example)

# ---------------------------------------
# 5. TRAIN THE NER MODEL
# ---------------------------------------

# Start the training process and get an optimizer object.
optimizer = nlp.begin_training()

# Train for 10 passes (epochs) over the training data.
for epoch in range(10):
    losses = {}

    # Create mini-batches of examples.
    # compounding(4.0, 32.0, 1.001) slowly grows the batch size from 4 to 32.
    batches = minibatch(examples, size=compounding(4.0, 32.0, 1.001))

    # For each batch, update the model.
    for batch in batches:
        # nlp.update() adjusts the model weights based on the examples.
        # drop=0.5 means 50% dropout (a regularization technique to reduce overfitting).
        nlp.update(batch, drop=0.5, losses=losses)

    # Print the loss after each epoch to see if the model is learning.
    print("Losses", losses)

# ---------------------------------------
# 6. TEST THE TRAINED MODEL
# ---------------------------------------

# Now use the trained model on a new sentence.
doc = nlp("I just bought a new iPhone.")

# Print the entities the model found in the text.
# Each entity has .text (the actual phrase) and .label_ (the entity type).
print("Named Entities:", [(ent.text, ent.label_) for ent in doc.ents])


This code demonstrates Dependency Parsing with spaCy. It loads a pre-trained English model, processes the sentence "The cat sat on the mat.", and then loops through each token to print its dependency role (such as subject, object, or root) and the head word it depends on. Finally, it uses displacy.render to visually display the dependency tree, letting you see how all the words in the sentence are grammatically connected.

In [ ]:
# Import the spaCy library for Natural Language Processing
import spacy

# ---------------------------------------
# 1. LOAD A PRE-TRAINED ENGLISH MODEL
# ---------------------------------------

# Load the small English model.
# This model knows how to split text into words, and how they relate to each other in a sentence.
# Make sure you have installed it first with:
#   python -m spacy download en_core_web_sm
nlp = spacy.load('en_core_web_sm')

# ---------------------------------------
# 2. SAMPLE TEXT
# ---------------------------------------

# This is the sentence we will analyze.
text = "The cat sat on the mat."

# ---------------------------------------
# 3. PROCESS THE TEXT
# ---------------------------------------

# Pass the text into the spaCy pipeline.
# The result "doc" contains tokens, part of speech tags, dependencies, etc.
doc = nlp(text)

# ---------------------------------------
# 4. PRINT DEPENDENCY PARSING RESULTS
# ---------------------------------------

print("Dependency Parsing:")

# Each token in "doc" is a word or punctuation mark.
for token in doc:
    # token.text  = the word itself
    # token.dep_  = the dependency label (how this word works in the sentence, like subject or object)
    # token.head  = the "head" word that this word depends on
    print(f"{token.text} ({token.dep_}): {token.head.text}")

# ---------------------------------------
# 5. VISUALIZE THE DEPENDENCY TREE
# ---------------------------------------

# displacy is a built in visualizer in spaCy
# It can draw the dependency tree so you can see how words are connected.
from spacy import displacy

# Render the dependency tree in a Jupyter Notebook or similar environment.
# style="dep" means we want to see dependency arcs.
# jupyter=True tells spaCy to display it directly in the notebook output.
displacy.render(doc, style="dep", jupyter=True)


This code demonstrates Training Custom Dependency Parsers with spaCy. It starts by creating a blank English pipeline and adding a dependency parser component. Then it defines a few custom dependency labels (like nsubj, dobj, and ROOT) and provides small training examples that specify, for each token, which word it depends on (heads) and what the relationship is (deps). Using these examples, the parser is trained over several epochs, with training loss printed to show learning progress. Finally, the trained model is tested on a new sentence, and the code prints each word along with its predicted dependency label and head word to show how the custom parser analyzes sentence structure.

In [ ]:
# Import spaCy, a powerful library for Natural Language Processing (NLP)
import spacy

# Import helper tools from spaCy:
# - DocBin: often used to store multiple Doc objects efficiently (not used directly here)
# - Example: wraps a Doc with its annotations for training
# - minibatch, compounding: help create mini-batches of training data
from spacy.tokens import DocBin
from spacy.training import Example
from spacy.util import minibatch, compounding

# ---------------------------------------
# 1. CREATE A BLANK ENGLISH PIPELINE
# ---------------------------------------

# Create a blank NLP pipeline for English.
# This means we start with no pre-trained components (just a tokenizer).
nlp = spacy.blank("en")

# ---------------------------------------
# 2. ADD A PARSER COMPONENT
# ---------------------------------------

# Add a dependency parser to the pipeline.
# The parser learns how words in a sentence are grammatically connected.
parser = nlp.add_pipe("parser")

# ---------------------------------------
# 3. DEFINE DEPENDENCY LABELS
# ---------------------------------------

# These are the dependency labels we want the parser to learn.
# In real projects, you would usually have more (and more precise) labels.
parser.add_label("nsubj")  # nominal subject
parser.add_label("dobj")   # direct object
parser.add_label("prep")   # preposition
parser.add_label("ROOT")   # root of the sentence (main verb)
parser.add_label("aux")    # auxiliary verb (helping verb)
parser.add_label("punct")  # punctuation

# ---------------------------------------
# 4. SAMPLE TRAINING DATA
# ---------------------------------------

# TRAIN_DATA is a list of (text, annotations) pairs.
# For dependency parsing, annotations contain:
#   - "heads": index of the head (governing word) for each token
#   - "deps":  dependency label for each token
#
# The length of "heads" and "deps" must match the number of tokens in the sentence.
TRAIN_DATA = [
    (
        "She enjoys playing tennis.",
        {
            "heads": [1, 1, 1, 3, 1],  # which token each word depends on
            "deps":  ["nsubj", "ROOT", "aux", "pobj", "punct"]  # relationship type
        }
    ),
    (
        "I like reading books.",
        {
            "heads": [1, 1, 2, 2, 1],
            "deps":  ["nsubj", "ROOT", "dobj", "punct", "punct"]
        }
    ),
]

# ---------------------------------------
# 5. CONVERT TRAINING DATA TO EXAMPLES
# ---------------------------------------

examples = []

# For each (text, annotations) pair:
for text, annotations in TRAIN_DATA:
    # Create a Doc object from the raw text using the pipeline's tokenizer.
    doc = nlp.make_doc(text)

    # Create an Example object that links the Doc with the annotations.
    # This tells spaCy what the correct heads and dependency labels should be.
    example = Example.from_dict(doc, annotations)

    # Add the example to our list of training examples.
    examples.append(example)

# ---------------------------------------
# 6. TRAIN THE PARSER
# ---------------------------------------

# Begin training and get an optimizer object.
optimizer = nlp.begin_training()

# Train for 10 epochs (passes over the training data).
for epoch in range(10):
    losses = {}

    # Create mini-batches from our examples.
    # compounding(4.0, 32.0, 1.001) gradually increases the batch size.
    batches = minibatch(examples, size=compounding(4.0, 32.0, 1.001))

    # For each batch, update the model's weights.
    for batch in batches:
        # nlp.update() adjusts the model according to the correct annotations.
        # drop=0.5 means 50% dropout (helps prevent overfitting).
        nlp.update(batch, drop=0.5, losses=losses)

    # Print how much error (loss) the model still has after this epoch.
    print("Losses", losses)

# ---------------------------------------
# 7. TEST THE TRAINED PARSER
# ---------------------------------------

# Use the trained model on a new sentence.
doc = nlp("She enjoys reading books.")

# For each token in the sentence, print:
# - the token text
# - its dependency label (dep_)
# - the head token it depends on
for token in doc:
    print(f"{token.text} ({token.dep_}): {token.head.text}")
